# 📓 Challenge — Pseudocódigo por Celdas
> Usá este archivo como guía para reescribir el notebook desde cero.
> Cada sección = una celda de Jupyter. El orden importa.

---

## 📋 Índice

1. [Setup & Carga](#celda-1--setup--carga-de-datos)
2. [Exploración Inicial](#celda-2--exploración-inicial)
3. [Detección del Momento Crítico](#celda-3--detección-del-momento-crítico)
4. [Diagnóstico dentro del Incidente](#celda-4--diagnóstico-dentro-del-incidente)
5. [Incidente vs Baseline](#celda-5--incidente-vs-baseline)
6. [Gráfico 1 — Severidad en el tiempo](#celda-6--gráfico-1--severidad-en-el-tiempo)
7. [Gráfico 2 — Bad Rate en el tiempo](#celda-7--gráfico-2--bad-rate-en-el-tiempo)
8. [BONUS — Análisis de Trace ID](#celda-8-bonus--análisis-de-trace-id)

---

## Celda 1 · Setup & Carga de datos

**¿Qué hace esta celda?**
Importa librerías, carga el CSV, convierte la columna de tiempo a datetime
y define la columna derivada `is_bad` que se va a usar en todo el notebook.

**Regla `is_bad`:**
Un evento es malo si cumple AL MENOS UNO de estos criterios:
- `severity` es `ERROR` o `CRITICAL`
- `status_code` es mayor o igual a 500

> ⚠️ No crear `is_5xx` ni ninguna columna extra. `is_bad` ya cubre ambos criterios.

```
IMPORTAR pandas como pd
IMPORTAR matplotlib.pyplot como plt

CARGAR el CSV desde la ruta del archivo → guardar en df

CONVERTIR la columna 'timestamp_event' a tipo datetime

CREAR columna booleana 'is_bad':
    es True si severity está en ['ERROR', 'CRITICAL']
    O si status_code >= 500

IMPRIMIR mensaje de confirmación
```

---

## Celda 2 · Exploración Inicial

**¿Qué hace esta celda?**
Responde las preguntas básicas del punto 6.1 del challenge usando tablas/prints.
No se filtra nada todavía, se trabaja sobre el `df` completo.

```
IMPRIMIR total de filas del df  (= total de logs)

IMPRIMIR la severidad más común:
    contar frecuencias de la columna 'severity'
    mostrar solo el top 1

IMPRIMIR el servicio con MÁS logs:
    contar frecuencias de 'service_name'
    mostrar el primero (mayor conteo)

IMPRIMIR el servicio con MENOS logs:
    del mismo conteo anterior
    mostrar el último (menor conteo)

IMPRIMIR el mensaje más repetido:
    contar frecuencias de 'message'
    mostrar solo el top 1

IMPRIMIR el mensaje "malo" más repetido:
    filtrar df donde is_bad == True
    contar frecuencias de 'message' en ese subset
    mostrar solo el top 1
```

---

## Celda 3 · Detección del Momento Crítico

**¿Qué hace esta celda?**
Agrupa todos los eventos en ventanas de 5 minutos, calcula métricas por ventana,
aplica el filtro de significancia y encuentra el momento con peor `bad_rate`.

**Definiciones que aplican aquí:**
- **Ventana:** bins de 5 minutos usando `timestamp_event`
- **bad_rate:** `bad_events / total_events` dentro de cada ventana
- **Momento crítico:** ventana con mayor `bad_rate`, con `total_events >= 20`

```
AGRUPAR df en ventanas de 5 minutos usando 'timestamp_event'
→ guardar el objeto de agrupación

AGREGAR por ventana las siguientes métricas:
    - total_events  → contar filas de la ventana
    - bad_events    → sumar la columna is_bad (True = 1)
    - avg_latency_ms → promedio de latency_ms
→ guardar como tabla 'analisis_temporal'

CALCULAR columna 'bad_rate':
    bad_events dividido total_events

FILTRAR ventanas con total_events >= 20
→ guardar como 'ventanas_validas'

ORDENAR 'ventanas_validas' por bad_rate de mayor a menor
TOMAR las primeras 5 filas → tabla top_5

IMPRIMIR la tabla top_5 con columnas:
    window_start | total_events | bad_events | bad_rate

EXTRAER el índice (timestamp) de la fila 0 del top_5
→ guardar como 'hora_critica'

IMPRIMIR el momento crítico exacto (hora_critica)
```

---

## Celda 4 · Diagnóstico dentro del Incidente

**¿Qué hace esta celda?**
Aísla exactamente los 5 minutos del incidente y responde:
qué servicio fue el más afectado, qué mensajes dominaron y qué endpoints
fueron los más comprometidos.

**Criterio declarado:** ranking por cantidad de bad events.

```
DEFINIR inicio_incidente = hora_critica
DEFINIR fin_incidente    = hora_critica + 5 minutos

FILTRAR df donde timestamp_event >= inicio_incidente
                  Y timestamp_event <  fin_incidente
→ guardar como df_incidente

FILTRAR df_incidente donde is_bad == True
→ guardar como df_errores_incidente

--- Ranking de servicios ---
CONTAR frecuencias de 'service_name' en df_errores_incidente
MOSTRAR top 10 (ranking de más a menos afectado)

--- Top mensajes de error ---
CONTAR frecuencias de 'message' en df_errores_incidente
MOSTRAR top 5

--- Top endpoints comprometidos ---
CONTAR frecuencias de 'endpoint' en df_errores_incidente
MOSTRAR top 5
```

---

## Celda 5 · Incidente vs Baseline

**¿Qué hace esta celda?**
Compara el período crítico contra el resto del tiempo (baseline)
en una tabla con métricas clave.

**Baseline:** todo el dataset FUERA del intervalo del incidente.

**Tabla resultante debe tener:**

| métrica | Incidente | Baseline |
|---|---|---|
| total_events | … | … |
| bad_rate | … | … |
| avg_latency_ms | … | … |

```
FILTRAR df donde timestamp_event ESTÁ FUERA del intervalo del incidente
    (antes de inicio_incidente) O (desde fin_incidente en adelante)
→ guardar como df_baseline

--- Métricas del incidente ---
EXTRAER la fila de 'hora_critica' desde analisis_temporal
→ tomar total_events, bad_rate, avg_latency_ms

--- Métricas del baseline ---
CALCULAR total de filas de df_baseline       → total_baseline
CALCULAR suma de is_bad en df_baseline       → bad_baseline
CALCULAR bad_rate del baseline:
    bad_baseline / total_baseline
CALCULAR promedio de latency_ms en df_baseline

CONSTRUIR un DataFrame de comparación con dos columnas:
    'Incidente' y 'Baseline'
    filas: total_events | bad_rate | avg_latency_ms

IMPRIMIR la tabla de comparación
```

---

## Celda 6 · Gráfico 1 — Severidad en el tiempo

**¿Qué hace esta celda?**
Visualiza cómo evolucionó cada nivel de severidad a lo largo del tiempo
en ventanas de 5 minutos. Permite ver visualmente cuándo explotaron
los ERROR y CRITICAL.

```
AGRUPAR df por [ventanas de 5min, 'severity']
CONTAR eventos por grupo
DESAPILAR 'severity' como columnas (unstack)
    → llenar huecos con 0 (fill_value=0)
→ guardar como conteo_severidad
    (índice = timestamps, columnas = INFO / WARN / ERROR / CRITICAL)

GRAFICAR conteo_severidad como líneas:
    - tamaño: 10 x 4
    - título: 'Severidad en el tiempo (5 min)'
    - eje X: 'Tiempo'
    - eje Y: 'Cantidad de eventos'
    - rotar etiquetas del eje X 45°

MOSTRAR gráfico
```

---

## Celda 7 · Gráfico 2 — Bad Rate en el tiempo

**¿Qué hace esta celda?**
Visualiza la evolución del `bad_rate` a lo largo del tiempo.
El pico debe coincidir visualmente con el momento crítico detectado.

> Usa `analisis_temporal` que ya fue calculado en la Celda 3.

```
CREAR figura de tamaño 10 x 4

GRAFICAR línea usando:
    - eje X: índice de analisis_temporal (timestamps de ventanas)
    - eje Y: columna 'bad_rate' de analisis_temporal

CONFIGURAR:
    - título: 'Bad Rate por ventana de 5 minutos'
    - eje X: 'Tiempo'
    - eje Y: 'Bad Rate'
    - rotar etiquetas del eje X 45°

MOSTRAR gráfico
```

---

## Celda 8 (BONUS) · Análisis de Trace ID

**¿Qué hace esta celda?**
Elige un `trace_id` que estuvo activo durante el incidente,
reconstruye la secuencia de eventos de ese trace y muestra
dónde apareció el primer bad event.

```
FILTRAR df_errores_incidente para obtener trace_ids presentes
CONTAR frecuencias de 'trace_id' en df_errores_incidente
TOMAR el trace_id más frecuente → guardar como trace_elegido

--- Reconstruir la traza completa ---
FILTRAR df donde trace_id == trace_elegido
ORDENAR por timestamp_event ascendente
SELECCIONAR columnas: timestamp_event | service_name | endpoint
                      | status_code | latency_ms | severity | message
→ guardar como df_traza

IMPRIMIR:
    "Trace ID analizado: {trace_elegido}"
IMPRIMIR df_traza completo

--- Primer bad event de la traza ---
FILTRAR df_traza donde is_bad == True
TOMAR la primera fila (iloc[0])

IMPRIMIR:
    "Primer bad event:"
    - timestamp
    - service_name
    - message
    - status_code
```

---

## 📝 Conclusiones (celda Markdown final)

Al final del notebook agregá una celda Markdown con las conclusiones.
Debe responder en 5-8 líneas:

```
- Momento crítico: [hora exacta de hora_critica]
- Servicio más afectado: [top 1 del ranking de servicios]
- Endpoint más comprometido: [top 1 del ranking de endpoints]
- Mensaje dominante: [mensaje malo más frecuente dentro del incidente]
- Comparación clave:
    bad_rate incidente: X.XX  vs  bad_rate baseline: X.XX
    avg_latency incidente: X ms  vs  avg_latency baseline: X ms
```

---

## ⚠️ Reglas del notebook

| Regla | Detalle |
|---|---|
| Sin `is_5xx` | No crear esa columna. `is_bad` ya cubre status >= 500 |
| Sin `%_5xx` | No incluir esa métrica en ninguna tabla |
| Orden de celdas | Las celdas deben poder correr de arriba a abajo sin errores |
| Variables clave | `hora_critica`, `inicio_incidente`, `fin_incidente` deben definirse y reutilizarse |
| `analisis_temporal` | Calculado una sola vez en Celda 3, usado en Celda 5 y Celda 7 |
| `reset_index()` | Usarlo después de groupby cuando necesites el timestamp como columna |
| Filtro de significancia | Siempre aplicar `total_events >= 20` antes de buscar el momento crítico |
